In [3]:
# GroupDNA - Week 1 minor project
# Name: Gaurang Garg / Batch: Data Science / Date: (21/09/26)

import numpy as np
from datetime import datetime, timedelta

filename = 'hostel_bois.txt'

## Feature 1: Parser

f = open(filename, 'r', encoding='utf-8')
content = f.read()
f.close()
lines = content.split('\n')

msgs = []
sys_count = 0
media_count = 0
del_count = 0
group_name = 'Group: CLG. ZOO'
print(group_name)


for line in lines:
    line = line.strip()
    if line == '':
        continue
    is_new = False
    if len(line) > 8 and line[2] == '/' and line[5] == '/':
        if line[0:2].isdigit() and line[3:5].isdigit() and line[6:8].isdigit():
            is_new = True

    if is_new == False:
        if len(msgs) > 0:
            msgs[-1]['text'] = msgs[-1]['text'] + ' ' + line
        continue

    t, rest = line.split(' - ', 1)
    if ': ' not in rest:
        sys_count += 1
        if 'created group' in rest:
            group_name = rest.split('"')[1]
        continue

    name, text = rest.split(': ', 1)
    kind = 'text'
    if text == '<Media omitted>':
        kind = 'media'
        media_count += 1
    if text == 'This message was deleted':
        kind = 'deleted'
        del_count += 1
    msgs.append({'time': datetime.strptime(t, '%d/%m/%y, %H:%M'),
                 'name': name, 'text': text, 'kind': kind})
assert len(msgs) > 0, 'no messages found in file'
typed = []
for m in msgs:
    if m['kind'] == 'text':
        typed.append(m)

print('parsed', len(msgs), 'messages | system:', sys_count, '| media:', media_count, '| deleted:', del_count)

## Feature 2 + 3: Overview, busiest day and hour

cnt = {}
for m in msgs:
    if m['name'] in cnt:
        cnt[m['name']] += 1
    else:
        cnt[m['name']] = 1


people = sorted(cnt, key=lambda x: cnt[x], reverse=True)
total = len(msgs)


first_day = msgs[0]['time'].date()
last_day = msgs[-1]['time'].date()
total_days = (last_day - first_day).days + 1

per_day = {}
for m in msgs:
    d = m['time'].date()
    per_day[d] = per_day.get(d, 0) + 1
best_day = max(per_day, key=lambda d: per_day[d])

def bar(v, mx):
    if v == 0:
        return '.'
    n = round(20 * v / mx)
    if n < 1:
        n = 1
    return '█' * n

## Feature 4: Heatmap (numpy)

heat = np.zeros((len(people), 24), dtype=int)
for m in msgs:
    r = people.index(m['name'])
    heat[r][m['time'].hour] += 1

hour_total = heat.sum(axis=0)
best_hour = int(hour_total.argmax())
def shade(v, mx):
    if mx == 0 or v == 0:
        return '.  '
    r = v / mx
    if r <= 0.25:
        return '.  '
    elif r <= 0.5:
        return '□  '
    elif r <= 0.75:
        return '★  '
    else:
        return '█  '

## Feature 5: Top words

stop = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'me', 'my',
        'you', 'hai', 'ho', 'ka', 'ki', 'ke', 'se', 'ko', 'ne', 'na', 'h', 'ye', 'wo', 'toh',
        'bhi', 'hi', 'aa', 'so', 'be', 'at', 'this', 'that', 'was', 'how', 'about', 'am', 'he',
        'his', 'she', 'her', 'we', 'they', 'with', 'have', 'has', 'had', 'are', 'but', 'not',
        'just', 'there', 'what', 'when', 'who', 'which', 'from', 'up', 'telling', 'everyone',
        'into', 'out', 'if', 'all', 'then', 'been', 'were', 'their', 'them', 'one', 'like',
        'because', 'would', 'could', 'really']
punct = '.,!?;:"\'()[]{}-_*/\\<>@#'
wc = {}
for m in typed:
    seen = []
    for w in m['text'].lower().split():
        w = w.strip(punct)
        if len(w) < 2 or w.isalpha() == False:
            continue
        if w in stop or w in seen:
            continue
        seen.append(w)
        wc[w] = wc.get(w, 0) + 1

top_words = sorted(wc.items(), key=lambda x: x[1], reverse=True)[:8]

## Feature 6: Response time and silent streaks

gaps = {}
for p in people:
    gaps[p] = []

for i in range(1, len(msgs)):
    if msgs[i]['name'] != msgs[i-1]['name']:
        diff = msgs[i]['time'] - msgs[i-1]['time']
        gaps[msgs[i]['name']].append(diff.total_seconds() / 60)

avg_resp = {}
for p in people:
    if len(gaps[p]) > 0:
        avg_resp[p] = sum(gaps[p]) / len(gaps[p])
    else:
        avg_resp[p] = 0

active_days = {}
for p in people:
    active_days[p] = set()
for m in msgs:
    active_days[m['name']].add(m['time'].date())

streak = {}
silent = {}
for p in people:
    longest = 0
    cur = 0
    end = None
    for i in range(total_days):
        day = first_day + timedelta(days=i)
        if day in active_days[p]:
            cur = 0
        else:
            cur += 1
            if cur > longest:
                longest = cur
                end = day
    streak[p] = (longest, end)
    silent[p] = total_days - len(active_days[p])

## Feature 6b: Instigator and Closer
# who sends the first message of the day, and who sends the last, most often

first_sender_by_day = {}
last_sender_by_day = {}
for m in msgs:
    d = m['time'].date()
    if d not in first_sender_by_day:
        first_sender_by_day[d] = m['name']
    last_sender_by_day[d] = m['name']

instigator_count = {}
closer_count = {}
for p in people:
    instigator_count[p] = 0
    closer_count[p] = 0
for d in first_sender_by_day:
    instigator_count[first_sender_by_day[d]] += 1
for d in last_sender_by_day:
    closer_count[last_sender_by_day[d]] += 1

instigator = max(instigator_count, key=lambda p: instigator_count[p])
closer = max(closer_count, key=lambda p: closer_count[p])

## Feature 6c: Longest message and each person's own busiest day

longest_msg = typed[0]
for m in typed:
    if len(m['text']) > len(longest_msg['text']):
        longest_msg = m

person_day_counts = {}
for p in people:
    person_day_counts[p] = {}
for m in msgs:
    d = m['time'].date()
    p = m['name']
    person_day_counts[p][d] = person_day_counts[p].get(d, 0) + 1

personal_best_day = {}
for p in people:
    best_d = max(person_day_counts[p], key=lambda d: person_day_counts[p][d])
    personal_best_day[p] = (best_d, person_day_counts[p][best_d])

## Feature 7: Archetypes

caring_list = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
               'reminder', 'drink water', "don't forget"]
laugh_list = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']


def spammer(name):
    runs = []
    c = 0
    for m in msgs:
        if m['name'] == name:
            c += 1
        else:
            if c > 0:
                runs.append(c)
            c = 0
    if c > 0:
        runs.append(c)
    return sum(runs) / len(runs)


def group_mom(name):
    s = 0
    for m in typed:
        if m['name'] == name:
            for k in caring_list:
                s += m['text'].lower().count(k)
    return s

def night_owl(name):
    n = 0
    tot = 0
    for m in msgs:
        if m['name'] == name:
            tot += 1
            if m['time'].hour >= 23 or m['time'].hour <= 4:
                n += 1
    return n / tot

def storyteller(name):
    words = 0
    c = 0
    for m in typed:
        if m['name'] == name:
            words += len(m['text'].split())
            c += 1
    return words / c
def drama_queen(name):
    n = 0
    c = 0
    for m in typed:
        if m['name'] == name:
            c += 1
            if (len(m['text']) >= 3 and m['text'].isupper()) or m['text'].count('!') >= 2:
                n += 1
    return n / c
def ghost(name):
    return silent[name] / total_days
def comedian(name):
    n = 0
    c = 0
    for m in typed:
        if m['name'] == name:
            c += 1
            for k in laugh_list:
                if k in m['text'].lower():
                    n += 1
                    break
    return n / c
def question_master(name):
    n = 0
    c = 0
    for m in typed:
        if m['name'] == name:
            c += 1
            if m['text'].endswith('?'):
                n += 1
    return n / c

raw = {}
for p in people:
    raw[p] = {'THE SPAMMER': spammer(p), 'THE GROUP MOM': group_mom(p),
              'THE NIGHT OWL': night_owl(p), 'THE STORYTELLER': storyteller(p),
              'THE DRAMA QUEEN': drama_queen(p), 'THE GHOST': ghost(p),
              'THE COMEDIAN': comedian(p), 'THE QUESTION MASTER': question_master(p)}

thresh = {'THE SPAMMER': 3, 'THE NIGHT OWL': 0.6, 'THE STORYTELLER': 30,
          'THE DRAMA QUEEN': 0.3, 'THE GHOST': 0.6, 'THE COMEDIAN': 0.1,
          'THE QUESTION MASTER': 0.25}
thresh['THE GROUP MOM'] = max([raw[p]['THE GROUP MOM'] for p in people])
if thresh['THE GROUP MOM'] == 0:
    thresh['THE GROUP MOM'] = 1

scores = []
for p in people:
    for a in raw[p]:
        scores.append((raw[p][a] / thresh[a], p, a))
scores.sort(reverse=True)
# tie break: highest score picks first, 1 archetype per person, 1 person per archetype.
# comedian + question master are only backups so they go in round 2
arch = {}
used = []
for rnd in [1, 2]:
    for s, p, a in scores:
        backup = (a == 'THE COMEDIAN' or a == 'THE QUESTION MASTER')
        if rnd == 1 and backup:
            continue
        if rnd == 2 and not backup:
            continue
        if p in arch or a in used:
            continue
        arch[p] = a
        used.append(a)


def proof(p):
    a = arch[p]
    r = raw[p][a]
    if a == 'THE SPAMMER':
        return f'avg {r:.1f} msgs in a row'
    elif a == 'THE GROUP MOM':
        return f'caring keyword score: {r}'
    elif a == 'THE NIGHT OWL':
        return f'{r*100:.1f}% msgs between 23h-04h'
    elif a == 'THE STORYTELLER':
        return f'avg {r:.1f} words per msg'
    elif a == 'THE DRAMA QUEEN':
        return f'{r*100:.1f}% ALL-CAPS / !! messages'
    elif a == 'THE GHOST':
        return f'silent on {silent[p]} of {total_days} days'
    elif a == 'THE COMEDIAN':
        return f'{r*100:.1f}% laugh messages'
    else:
        return f"{r*100:.1f}% end with '?'"

## Feature 8: Final report

line = '=' * 60
print(line)
print(f'  GROUPDNA REPORT — "{group_name}"')
print(f'  {total_days} days • {total:,} messages • {len(people)} members')
print(line)
print(f"  Period      : {msgs[0]['time']:%d %B %Y} to {msgs[-1]['time']:%d %B %Y}")
print(f'  Busiest day : {best_day:%d %B %Y} ({per_day[best_day]} messages)')
print(f'  Busiest hour: {best_hour:02d}:00 - {best_hour+1:02d}:00 ({int(hour_total[best_hour])} messages)')
print(f'  Skipped     : {sys_count} system | {media_count} media | {del_count} deleted')
print('\n  MESSAGES PER PERSON')
for p in people:
    pct = cnt[p] / total * 100
    print(f'  {p:<8}{bar(cnt[p], cnt[people[0]]):<21}{cnt[p]:>4} ({pct:4.1f}%)')


print('\n  ACTIVITY HEATMAP (columns = hour 00 to 23)')
head = '  ' + ' ' * 8 + ' '
for h in range(24):
    head += f'{h:02d} '
print(head)
for i in range(len(people)):
    row = ''
    for v in heat[i]:
        row += shade(int(v), int(heat[i].max()))
    print(f'  {people[i]:<8} {row}  ({int(heat[i].sum())})')

print("\n  THIS GROUP'S FAVOURITE WORDS")
for w, c in top_words:
    print(f'  {w:<10}{bar(c, top_words[0][1]):<21}{c}')


print('\n  RESPONSE PATTERNS')
ok = [p for p in people if avg_resp[p] > 0]
fast = min(ok, key=lambda p: avg_resp[p])
slow = max(ok, key=lambda p: avg_resp[p])
for label, p in [('Fastest', fast), ('Slowest', slow)]:
    mins = avg_resp[p]
    if mins < 60:
        t = f'{mins:.1f} minutes'
    else:
        t = f'{mins/60:.1f} hours'
    print(f'  {label} replier : {p} (avg {t})')
print('\n  LONGEST SILENT STREAKS')
for p in sorted(people, key=lambda x: streak[x][0], reverse=True):
    n, end = streak[p]
    extra = ''
    if n > 0:
        extra = f' ({end - timedelta(days=n-1):%d %b} - {end:%d %b})'
    print(f'  {p:<8}: {n} {"day" if n == 1 else "days"}{extra}')

print('\n  PERSONALITY ARCHETYPES')
for p in people:
    print(f'  {p:<8}→ {arch[p]:<20} ({proof(p)})')
print(f'  {instigator:<8}→ {"THE INSTIGATOR":<20} (starts the day {instigator_count[instigator]} of {total_days} days)')
print(f'  {closer:<8}→ {"THE CLOSER":<20} (ends the day {closer_count[closer]} of {total_days} days)')

print('\n  LONGEST MESSAGE EVER SENT')
preview = longest_msg['text']
if len(preview) > 120:
    preview = preview[:120] + '...'
print(f'  {longest_msg["name"]} ({len(longest_msg["text"])} characters):')
print(f'  "{preview}"')

print("\n  EACH PERSON'S OWN BUSIEST DAY")
for p in people:
    d, c = personal_best_day[p]
    print(f'  {p:<8}: {d:%d %B %Y} ({c} messages)')

print('\n' + line)
print('  Generated by GroupDNA • Built with Python + NumPy')
print(line)

Group: CLG. ZOO
parsed 3174 messages | system: 4 | media: 32 | deleted: 15
  GROUPDNA REPORT — "Hostel Bois 4ever"
  60 days • 3,174 messages • 6 members
  Period      : 01 April 2024 to 30 May 2024
  Busiest day : 04 May 2024 (76 messages)
  Busiest hour: 18:00 - 19:00 (248 messages)
  Skipped     : 4 system | 32 media | 15 deleted

  MESSAGES PER PERSON
  Rahul   ████████████████████  953 (30.0%)
  Priya   ███████████████       718 (22.6%)
  Neha    █████████████         635 (20.0%)
  Aman    ██████████            490 (15.4%)
  Karan   ███████               354 (11.2%)
  Vikas   █                      24 ( 0.8%)

  ACTIVITY HEATMAP (columns = hour 00 to 23)
           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
  Rahul    .  .  .  .  .  .  .  .  .  .  .  .  ★  □  □  ★  ★  □  █  ★  □  █  ★  ★    (953)
  Priya    .  .  .  .  .  .  .  □  ★  █  █  █  █  ★  ★  □  □  ★  ★  █  ★  □  □  .    (718)
  Neha     .  .  .  .  .  □  .  .  ★  █  █  □  ★  ★  □  .  ★  █  █